<a href="https://colab.research.google.com/github/CrisHayashi/CrisHayashi/blob/main/Loginveste_planilha_com_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# from google.colab import files
# uploaded = files.upload()

# EXTRAÇAO
src="/content/fin2024.xlsx"
df = pd.read_excel(src, sheet_name="sheet1")
df = df.copy()  # boa prática para evitar warnings

# df.head()
# df.keys()

# Mostrar resumo inicial
print("Linhas:", len(df))
print("Colunas:", df.columns.tolist())
display(df.head(8))

Linhas: 1100
Colunas: ['Fornecedor', 'No NF', 'Valor do Doc', 'Data Pagto.', 'Centro de Custos', 'Categoria', 'Banco', 'Cheque Nº', 'Competencia', 'OBS']


,Fornecedor,No NF,Valor do Doc,Data Pagto.,Centro de Custos,Categoria,Banco,Cheque Nº,Competencia,OBS
0,PETRO CABO,NaN,120.00,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Manutencao e conservacao,BRADESCO,000016,1,Gasolina p/ roçadeira
1,DISMACON COMERCIO DE MATERIAIS DE CONSTRUÇÃO LTDA,NaN,23.37,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Materiais e suprimentos,BRADESCO,000016,1,Compra de fita crepe
2,PANIFICADORA 5 ESTRELAS,NaN,15.00,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Manutencao e conservacao,BRADESCO,000016,1,Compra de sacos de Nylon
3,J R AGROPECUARIA LTDA,908,557.79,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Manutencao e conservacao,BRADESCO,000016,1,Compra de nylon e bobina p/ a roçadeira
4,LOGNET COM E TECNOLOGIA,NaN,62.68,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Patrimonial,BRADESCO,000016,1,01 mouse p/ a recepção
5,CIA BRASILEIRA DE DISTRIBUIÇÃO - EXTRA,NaN,794.69,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Despesas administrativas,BRADESCO,000016,1,Compra de panetone p/ os colaboradores
6,GUARARAPES PARKING LTDA,NaN,11.50,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Despesas administrativas,BRADESCO,000016,1,Estacionamento p/ compra dos panetones
7,RONALDO CARNEIRO DA SILVA JUNIOR,267,265.00,2024-01-02,05.10 DA - Despesas com Fundo de Caixa,Manutencao e conservacao,BRADESCO,000016,1,Conserto da roçadeira


In [6]:
# Parse de datas (com fallback)

# Tentar converter direto (dia primeiro já que estamos no Brasil mas nao funcionou)
df["data_pagto_parsed"] = pd.to_datetime(df.get("Data Pagto.", ""), dayfirst=True, errors="coerce")

# Se houver NaT, tentar formatos adicionais comuns
mask_nat = df["data_pagto_parsed"].isna()
if mask_nat.any():
    formatos = ["%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M:%S", "%d-%m-%Y %H:%M", "%Y/%m/%d"]
    for fmt in formatos:
        df.loc[mask_nat, "data_pagto_parsed"] = pd.to_datetime(
            df.loc[mask_nat, "Data Pagto."],
            format=fmt,
            errors="coerce"
        )
        mask_nat = df["data_pagto_parsed"].isna()
        if not mask_nat.any():
            break

# Normalizar (zerar horas/minutos/segundos)
df["data_pagto_norm"] = df["data_pagto_parsed"].dt.normalize()

display(df[["Data Pagto.","data_pagto_parsed","data_pagto_norm"]].head(8))

,Data Pagto.,data_pagto_parsed,data_pagto_norm
0,2024-01-02,2024-01-02,2024-01-02
1,2024-01-02,2024-01-02,2024-01-02
2,2024-01-02,2024-01-02,2024-01-02
3,2024-01-02,2024-01-02,2024-01-02
4,2024-01-02,2024-01-02,2024-01-02
5,2024-01-02,2024-01-02,2024-01-02
6,2024-01-02,2024-01-02,2024-01-02
7,2024-01-02,2024-01-02,2024-01-02


In [7]:
# Criar dim_Tempo

# Datas únicas ordenadas
unique_dates = df["data_pagto_norm"].dropna().drop_duplicates().sort_values().reset_index(drop=True)

# Criar dataframe de dimensão
dim_Tempo = pd.DataFrame({"Data": unique_dates})

# Criar surrogate key sequencial
dim_Tempo["ID_DataPagamento"] = range(1, len(dim_Tempo) + 1)

# Colunas derivadas
dim_Tempo["Ano"] = dim_Tempo["Data"].dt.year
dim_Tempo["Mes"] = dim_Tempo["Data"].dt.month
dim_Tempo["Dia"] = dim_Tempo["Data"].dt.day

# Adicionar linha "Unknown"
dim_Tempo = pd.concat([
    pd.DataFrame([{
        "Data": pd.NaT,
        "ID_DataPagamento": 0,
        "Ano": None,
        "Mes": None,
        "Dia": None}]),
    dim_Tempo
], ignore_index=True)

# Coluna data formatada em DDMMAAAA (string)
dim_Tempo["Data_Formatada"] = dim_Tempo["Data"].dt.strftime("%d%m%Y")

# Tratar NaT do "Unknown"
dim_Tempo["Data_Formatada"] = dim_Tempo["Data_Formatada"].fillna("Unknown")

# Reordenar colunas
dim_Tempo = dim_Tempo[[
    "ID_DataPagamento", "Data", "Ano", "Mes", "Dia", "Data_Formatada"
]]

# Remover colunas duplicadas antes do merge
df = df.drop(columns=["Data", "ID_DataPagamento"], errors="ignore")

# Mapear id_tempo de volta ao df original
df = df.merge(
    dim_Tempo[["Data", "ID_DataPagamento"]],
    left_on="data_pagto_norm",
    right_on="Data",
    how="left"
)
df["ID_DataPagamento"] = df["ID_DataPagamento"].fillna(0).astype(int)

# Visualizar as primeiras linhas da Dim_Tempo
print(dim_Tempo.head(10).to_string(index=False))

 ID_DataPagamento       Data  Ano  Mes  Dia Data_Formatada
                0        NaT None None None        Unknown
                1 2024-01-02 2024    1    2       02012024
                2 2024-01-10 2024    1   10       10012024
                3 2024-01-11 2024    1   11       11012024
                4 2024-01-15 2024    1   15       15012024
                5 2024-01-22 2024    1   22       22012024
                6 2024-01-26 2024    1   26       26012024
                7 2024-01-30 2024    1   30       30012024
                8 2024-02-06 2024    2    6       06022024
                9 2024-02-08 2024    2    8       08022024


In [8]:
# Criar coluna tipo_pagamento
df["TipoPagamento"] = np.where(df["Cheque Nº"].astype(str).str.isnumeric(), "Cheque", df["Cheque Nº"])

# Conferir resultado
print(df[["Cheque Nº", "TipoPagamento"]].head(10))

    Cheque Nº TipoPagamento
0      000016        Cheque
1      000016        Cheque
2      000016        Cheque
3      000016        Cheque
4      000016        Cheque
5      000016        Cheque
6      000016        Cheque
7      000016        Cheque
8  PAGTO ELET    PAGTO ELET
9  PAGTO ELET    PAGTO ELET


In [9]:
# Criar dim_TipoPagamento

# Garantir string e criar coluna Cheque No
df["Cheque No"] = df["Cheque Nº"].astype(str).str.strip()

# Criar coluna TipoPagamento
df["TipoPagamento"] = np.where(
    df["Cheque No"].str.isnumeric(),
    "Cheque",                  # Se só tiver dígitos → é cheque
    df["Cheque No"]            # Se não for número → mantém valor (ex: PIX, TED, Dinheiro)
)

# Conferir resultado parcial
print("Prévia do mapeamento Cheque Nº → TipoPagamento")
print(df[["Cheque No", "TipoPagamento"]].head(10).to_string(index=False))

# Criar Dim_TipoPagamento com valores únicos
dim_TipoPagamento = pd.DataFrame({
    "Nome_Tipo_Pagto": df["TipoPagamento"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Criar surrogate key sequencial
dim_TipoPagamento["ID_TipoPagamento"] = range(1, len(dim_TipoPagamento) + 1)

# Adicionar linha Unknown
dim_TipoPagamento = pd.concat([
    pd.DataFrame([{"ID_TipoPagamento": 0, "Nome_Tipo_Pagto": "Unknown"}]),
    dim_TipoPagamento
], ignore_index=True)

# Reordenar colunas para consistência
dim_TipoPagamento = dim_TipoPagamento[["ID_TipoPagamento", "Nome_Tipo_Pagto"]]

# Mapear ID_TipoPagamento de volta ao df
df = df.merge(
    dim_TipoPagamento,
    left_on="TipoPagamento",
    right_on="Nome_Tipo_Pagto",
    how="left"
)

# Garantir tipo inteiro
df["ID_TipoPagamento"] = df["ID_TipoPagamento"].fillna(0).astype(int)

print("\nPrévia da Base Pagamentos (10 primeiras linhas):")
print(df[["Cheque No", "TipoPagamento", "ID_TipoPagamento"]].head(10).to_string(index=False))

print("\nDimensão TipoPagamento:")
print(dim_TipoPagamento.head(10).to_string(index=False))

Prévia do mapeamento Cheque Nº → TipoPagamento
 Cheque No TipoPagamento
    000016        Cheque
    000016        Cheque
    000016        Cheque
    000016        Cheque
    000016        Cheque
    000016        Cheque
    000016        Cheque
    000016        Cheque
PAGTO ELET    PAGTO ELET
PAGTO ELET    PAGTO ELET

Prévia da Base Pagamentos (10 primeiras linhas):
 Cheque No TipoPagamento  ID_TipoPagamento
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
    000016        Cheque                 1
PAGTO ELET    PAGTO ELET                 3
PAGTO ELET    PAGTO ELET                 3

Dimensão TipoPagamento:
 ID_TipoPagamento Nome_Tipo_Pagto
                0         Unknown
                1          Cheque
                2         DE

In [10]:
# Criar Dim_Fornecedor com valores únicos
dim_Fornecedor = pd.DataFrame({
    "Nome_Fornecedor": df["Fornecedor"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Criar surrogate key sequencial
dim_Fornecedor["ID_Fornecedor"] = range(1, len(dim_Fornecedor) + 1)

# Adicionar linha Unknown
dim_Fornecedor = pd.concat([
    pd.DataFrame([{"ID_Fornecedor": 0, "Nome_Fornecedor": "Unknown"}]),
    dim_Fornecedor
], ignore_index=True)

# Reordenar colunas (ID primeiro)
dim_Fornecedor = dim_Fornecedor[["ID_Fornecedor", "Nome_Fornecedor"]]

# Evitar conflito antes do merge
df = df.drop(columns=["ID_Fornecedor", "Nome_Fornecedor"], errors="ignore")

# Mapear ID_Fornecedor de volta ao df
df = df.merge(
    dim_Fornecedor,
    left_on="Fornecedor",
    right_on="Nome_Fornecedor",
    how="left"
)

# Garantir tipo inteiro
df["ID_Fornecedor"] = df["ID_Fornecedor"].fillna(0).astype(int)

# Conferir resultado
print("\nPrévia da base Fornecedor (10 primeiras linhas):")
print(df[["ID_Fornecedor", "Fornecedor"]].head(10).to_string(index=False))

print("\nDimensão Fornecedor:")
print(dim_Fornecedor.head(10).to_string(index=False))


Prévia da base Fornecedor (10 primeiras linhas):
 ID_Fornecedor                                        Fornecedor
           112                                        PETRO CABO
            35 DISMACON COMERCIO DE MATERIAIS DE CONSTRUÇÃO LTDA
           109                           PANIFICADORA 5 ESTRELAS
            73                             J R AGROPECUARIA LTDA
            88                           LOGNET COM E TECNOLOGIA
            24            CIA BRASILEIRA DE DISTRIBUIÇÃO - EXTRA
            66                           GUARARAPES PARKING LTDA
           132                  RONALDO CARNEIRO DA SILVA JUNIOR
            23                                             CELPE
           127      PTC SERVIÇOS E MONIT DE SIST DE SEGURANÇA ME

Dimensão Fornecedor:
 ID_Fornecedor                                     Nome_Fornecedor
             0                                             Unknown
             1                                 A M VIDRAÇARIA LTDA
            

In [11]:
# Criar Dim_Competencia
dim_Competencia = pd.DataFrame({
    "Mes_Competencia": df["Competencia"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

dim_Competencia["ID_Competencia"] = range(1, len(dim_Competencia) + 1)

# Adicionar linha Unknown
dim_Competencia = pd.concat([
    pd.DataFrame([{"ID_Competencia": 0, "Mes_Competencia": "Unknown"}]),
    dim_Competencia
], ignore_index=True)

# Reordenar colunas
dim_Competencia = dim_Competencia[["ID_Competencia", "Mes_Competencia"]]

# Evitar conflito antes do merge
df = df.drop(columns=["ID_Competencia", "Mes_Competencia"], errors="ignore")

# Mapear idCompetencia de volta ao df
df = df.merge(
    dim_Competencia,
    left_on="Competencia",
    right_on="Mes_Competencia",
    how="left"
)

# Garantir tipo inteiro
df["ID_Competencia"] = df["ID_Competencia"].fillna(0).astype(int)

# Conferir resultado
print("\nDimensão Competencia:")
print(dim_Competencia.head(10).to_string(index=False))


Dimensão Competencia:
 ID_Competencia Mes_Competencia
              0         Unknown
              1               1
              2               2
              3               3
              4               4
              5               5
              6               6
              7               7
              8               8
              9               9


In [12]:
# Criar Dim_Categoria
dim_Categoria = pd.DataFrame({
    "categoria": df["Categoria"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

dim_Categoria["ID_Categoria"] = range(1, len(dim_Categoria) + 1)

# Adicionar linha Unknown
dim_Categoria = pd.concat([
    pd.DataFrame([{"ID_Categoria": 0, "categoria": "Unknown"}]),
    dim_Categoria
], ignore_index=True)

# Reordenar colunas
dim_Categoria = dim_Categoria[["ID_Categoria", "categoria"]]

# Mapear idCategoria de volta ao df
df = df.merge(
    dim_Categoria,
    left_on="Categoria", # usa a coluna original do df
    right_on="categoria",
    how="left"
)

# Garantir tipo inteiro
df["ID_Categoria"] = df["ID_Categoria"].fillna(0).astype(int)

# Conferir resultado
print("\nDimensão Categoria:")
print(dim_Categoria.head(10).to_string(index=False))


Dimensão Categoria:
 ID_Categoria                    categoria
            0                      Unknown
            1     Despesas administrativas
            2         Despesas com pessoal
            3 Encargos e obrigacoes legais
            4     Manutencao e conservacao
            5      Materiais e Suprimentos
            6      Materiais e suprimentos
            7                       Outros
            8                  Patrimonial
            9                    Seguranca


In [13]:
import re

# Criar coluna limpa (remover códigos e siglas antes do nome do centro de custos)
df["Centro_Custo_Limpo"] = df["Centro de Custos"].astype(str).apply(
    lambda x: re.sub(r"^\d+(\.\d+)?\s*([A-Z]{0,5})?\s*-?\s*", "", x).strip()
)

# Criar Dim_CentroCustos com valores únicos
dim_CentroCustos = pd.DataFrame({
    "Centro_Custo": df["Centro_Custo_Limpo"].dropna().drop_duplicates().sort_values().reset_index(drop=True)
})

# Criar surrogate key sequencial
dim_CentroCustos["ID_CentroCustos"] = range(1, len(dim_CentroCustos) + 1)

# Adicionar linha Unknown
dim_CentroCustos = pd.concat([
    pd.DataFrame([{"ID_CentroCustos": 0, "Centro_Custo": "Unknown"}]),
    dim_CentroCustos
], ignore_index=True)

# Reordenar colunas (ID primeiro)
dim_CentroCustos = dim_CentroCustos[["ID_CentroCustos", "Centro_Custo"]]

# Evitar conflito antes do merge
df = df.drop(columns=["ID_CentroCustos", "Centro_Custo"], errors="ignore")

# Mapear ID_CentroCustos de volta ao df
df = df.merge(
    dim_CentroCustos,
    left_on="Centro_Custo_Limpo",
    right_on="Centro_Custo",
    how="left"
)

# Garantir tipo inteiro
df["ID_CentroCustos"] = df["ID_CentroCustos"].fillna(0).astype(int)

# Conferir resultado
print("\nPrévia da base Centro de Custos (10 primeiras linhas):")
print(df[["ID_CentroCustos", "Centro de Custos", "Centro_Custo_Limpo"]].head(10).to_string(index=False))

print("\nDimensão CentroCustos:")
print(dim_CentroCustos.head(10).to_string(index=False))


Prévia da base Centro de Custos (10 primeiras linhas):
 ID_CentroCustos                       Centro de Custos          Centro_Custo_Limpo
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              11 05.10 DA - Despesas com Fundo de Caixa Despesas com Fundo de Caixa
              13    02.02 DO - Energia Elétrica - CELPE    Energia Elétrica - CELPE
              22        03.09 DT - Locação de Rádios HT        Locação de Rádios HT

Dimensão CentroCust

In [14]:
# Tratar campos vazios no No NF
df["No NF"] = df.get("No NF", "").fillna("Sem NF")

print(df[["Fornecedor", "No NF", "Valor do Doc"]].head(10))

                                          Fornecedor   No NF  Valor do Doc
0                                         PETRO CABO  Sem NF        120.00
1  DISMACON COMERCIO DE MATERIAIS DE CONSTRUÇÃO LTDA  Sem NF         23.37
2                            PANIFICADORA 5 ESTRELAS  Sem NF         15.00
3                              J R AGROPECUARIA LTDA     908        557.79
4                            LOGNET COM E TECNOLOGIA  Sem NF         62.68
5             CIA BRASILEIRA DE DISTRIBUIÇÃO - EXTRA  Sem NF        794.69
6                            GUARARAPES PARKING LTDA  Sem NF         11.50
7                   RONALDO CARNEIRO DA SILVA JUNIOR     267        265.00
8                                              CELPE  Sem NF      92536.25
9       PTC SERVIÇOS E MONIT DE SIST DE SEGURANÇA ME    1318        142.50


In [15]:
# montar Fato_Pagamentos agregando por todas as dimensões

Fato_Pagamentos = df.groupby([
    "ID_Fornecedor",
    "ID_DataPagamento",
    "ID_Competencia",
    "ID_TipoPagamento",
    "ID_Categoria",
    "ID_CentroCustos"
], as_index=False).agg(
    valorDocumento=("Valor do Doc", "sum"),   # soma dos valores pagos
    qtdPagamentos=("Valor do Doc", "count")   # quantidade de pagamentos realizados
).copy()  # <- aqui criamos uma cópia independente

display(Fato_Pagamentos.head(8))


,ID_Fornecedor,ID_DataPagamento,ID_Competencia,ID_TipoPagamento,ID_Categoria,ID_CentroCustos,valorDocumento,qtdPagamentos
0,1,120,11,3,4,11,120.00,1
1,2,43,5,1,1,11,46.00,1
2,3,93,9,3,4,23,5360.00,1
3,4,60,6,1,1,27,780.00,1
4,5,2,1,3,10,33,2508.82,1
5,5,2,1,3,10,38,37304.77,1
6,5,9,2,3,10,38,37304.77,1
7,5,10,2,3,10,33,2688.00,1


In [17]:
# ======================================
# ETL Final - Dimensões + Resumo
# ======================================

# --- Dim Tempo  ---
display(dim_Tempo.head(5))

# --- Dim Fornecedor ---
display(dim_Fornecedor.head(5))

# --- Dim TipoPagamento ---
display(dim_TipoPagamento.head(5))

# --- Dim CentroCustos ---
display(dim_CentroCustos.head(5))

# --- Dim Categoria ---
display(dim_Categoria.head(5))

# --- Dim Competencia ---
display(dim_Competencia.head(5))

# --- Resumo Global ---
totalPagamentos = df.shape[0]
valorTotalPagamentos = df["Valor do Doc"].sum()

# Formatar valor total com separador de milhar e duas casas decimais
resumoPagamentos = pd.DataFrame({
    "totalPagamentos": [totalPagamentos],
    "valorTotalPagamentos": [f"{valorTotalPagamentos:,.2f}"]
})

print("Resumo global de pagamentos:")
display(resumoPagamentos)

,ID_DataPagamento,Data,Ano,Mes,Dia,Data_Formatada
0,0,NaT,None,None,None,Unknown
1,1,2024-01-02,2024,1,2,02012024
2,2,2024-01-10,2024,1,10,10012024
3,3,2024-01-11,2024,1,11,11012024
4,4,2024-01-15,2024,1,15,15012024


,ID_Fornecedor,Nome_Fornecedor
0,0,Unknown
1,1,A M VIDRAÇARIA LTDA
2,2,AC MORAIS COMBUSTIVEIS LTDA - IPIRANGA
3,3,ACUMULADORES DE ENERGIA CENTRAL DAS BATERIAS E...
4,4,AF PRINT GRAFICA


,ID_TipoPagamento,Nome_Tipo_Pagto
0,0,Unknown
1,1,Cheque
2,2,DEB C/C
3,3,PAGTO ELET
4,4,TED


,ID_CentroCustos,Centro_Custo
0,0,Unknown
1,1,AVCB
2,2,Agua Mineral
3,3,Análise da Água
4,4,Auxiliar de Patrimonio


,ID_Categoria,categoria
0,0,Unknown
1,1,Despesas administrativas
2,2,Despesas com pessoal
3,3,Encargos e obrigacoes legais
4,4,Manutencao e conservacao


,ID_Competencia,Mes_Competencia
0,0,Unknown
1,1,1
2,2,2
3,3,3
4,4,4


Resumo global de pagamentos:


,totalPagamentos,valorTotalPagamentos
0,1100,"4,306,209.66"
